# Orbital Mechanics
*From Kepler's laws to interplanetary transfers — with live simulations*

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from ipywidgets import interact, IntSlider, FloatSlider
from IPython.display import display, clear_output
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
})

# --- constants ---
G   = 6.67430e-11       # m^3 kg^-1 s^-2
MU  = 3.986004418e14    # Earth  GM  m^3/s^2
MU_S = 1.32712440018e20 # Sun    GM  m^3/s^2
R_E = 6.371e6           # Earth radius  m
G0  = 9.80665           # m/s^2
J2  = 1.08263e-3        # Earth J2
AU  = 1.496e11          # m
OMEGA_E = 7.2921159e-5  # Earth rotation rate  rad/s

rows = [
    ('G',               f'{G:.5e}',              'm³ kg⁻¹ s⁻²'),
    ('μ_Earth',         f'{MU:.4e}',             'm³/s²'),
    ('μ_Sun',           f'{MU_S:.5e}',           'm³/s²'),
    ('R_Earth',         f'{R_E/1e3:.1f}',        'km'),
    ('J2',              f'{J2:.5e}',             '—'),
    ('1 AU',            f'{AU/1e9:.3f}',         '10⁹ m'),
    ('Ω_Earth',         f'{OMEGA_E:.5e}',        'rad/s'),
]
print(f"{'Constant':<20} {'Value':>16}  Unit")
print('-' * 50)
for name, val, unit in rows:
    print(f'{name:<20} {val:>16}  {unit}')

Constant                        Value  Unit
--------------------------------------------------
G                         6.67430e-11  m³ kg⁻¹ s⁻²
μ_Earth                    3.9860e+14  m³/s²
μ_Sun                     1.32712e+20  m³/s²
R_Earth                        6371.0  km
J2                        1.08263e-03  —
1 AU                          149.600  10⁹ m
Ω_Earth                   7.29212e-05  rad/s


## 1 — Kepler's Laws: The Grammar of Orbits

**First law** — every orbit is a conic section (ellipse, parabola, hyperbola) with the central body at one focus.

The polar equation of an ellipse:

$$r(\theta) = \frac{a(1 - e^2)}{1 + e\cos\theta}$$

**Second law** — the radius vector sweeps *equal areas in equal times*.

**Third law** — the square of the period is proportional to the cube of the semi-major axis:

$$T^2 = \frac{4\pi^2}{\mu}\,a^3$$

| Symbol | Meaning |
|:--|:--|
| $a$ | Semi-major axis |
| $e$ | Eccentricity (0 = circle, 0–1 = ellipse) |
| $\theta$ | True anomaly — angle from periapsis |
| $\mu$ | Gravitational parameter $GM$ |

In [2]:
def kepler_E(M, e, tol=1e-12):
    """Solve Kepler's equation M = E - e sin(E) via Newton-Raphson."""
    E = M + 0.85 * e * np.sign(np.sin(M))
    for _ in range(50):
        dE = (E - e * np.sin(E) - M) / (1 - e * np.cos(E))
        E -= dE
        if np.all(np.abs(dE) < tol):
            break
    return E

def true_from_mean(M, e):
    E = kepler_E(M, e)
    return 2 * np.arctan2(np.sqrt(1 + e) * np.sin(E / 2),
                          np.sqrt(1 - e) * np.cos(E / 2))

out1 = widgets.Output()
ecc_sl = widgets.FloatSlider(value=0.5, min=0.0, max=0.95, step=0.01,
    description='Eccentricity', style={'description_width': '110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
a_sl = widgets.FloatSlider(value=20000, min=7000, max=50000, step=500,
    description='a (km)', style={'description_width': '110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)

def update_kepler(change=None):
    e = ecc_sl.value
    a = a_sl.value * 1e3
    T = 2 * np.pi * np.sqrt(a**3 / MU)
    theta = np.linspace(0, 2 * np.pi, 800)
    r = a * (1 - e**2) / (1 + e * np.cos(theta))
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    M_arr = np.linspace(0, 2 * np.pi, 600)
    nu_arr = true_from_mean(M_arr, e)
    t_arr = M_arr / (2 * np.pi) * T
    # equal-area wedges
    dt_wedge = T / 8
    M1_start, M1_end = 0.0, 2 * np.pi * dt_wedge / T
    M2_start = np.pi
    M2_end = M2_start + 2 * np.pi * dt_wedge / T
    def wedge_coords(Ms, Me, n=80):
        M_w = np.linspace(Ms, Me, n)
        nu_w = true_from_mean(M_w, e)
        r_w = a * (1 - e**2) / (1 + e * np.cos(nu_w))
        xw = np.concatenate([[0], r_w * np.cos(nu_w), [0]])
        yw = np.concatenate([[0], r_w * np.sin(nu_w), [0]])
        return xw, yw
    w1x, w1y = wedge_coords(M1_start, M1_end)
    w2x, w2y = wedge_coords(M2_start, M2_end)
    r_peri = a * (1 - e)
    r_apo = a * (1 + e)
    v_peri = np.sqrt(MU * (2 / r_peri - 1 / a))
    v_apo = np.sqrt(MU * (2 / r_apo - 1 / a))
    with out1:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
        ax1.plot(x / 1e6, y / 1e6, 'k', lw=1.5)
        ax1.fill(w1x / 1e6, w1y / 1e6, alpha=0.3, color='C0', label='Area 1 (near periapsis)')
        ax1.fill(w2x / 1e6, w2y / 1e6, alpha=0.3, color='C3', label='Area 2 (near apoapsis)')
        ax1.plot(0, 0, 'o', color='#1a6fad', ms=14, zorder=5)
        ax1.plot(r_peri / 1e6, 0, 'o', color='C2', ms=8, zorder=5)
        ax1.plot(-r_apo / 1e6, 0, 'o', color='C3', ms=8, zorder=5)
        # velocity arrows
        arrow_scale = max(r_apo / 1e6, 1) * 0.15
        ax1.annotate('', xy=(r_peri / 1e6, arrow_scale * v_peri / v_peri),
                     xytext=(r_peri / 1e6, 0),
                     arrowprops=dict(arrowstyle='->', color='C2', lw=2))
        ax1.text(r_peri / 1e6 + 0.5, arrow_scale * 0.5,
                 f'v_p={v_peri/1e3:.2f} km/s', fontsize=8, color='C2')
        ax1.annotate('', xy=(-r_apo / 1e6, -arrow_scale * v_apo / v_peri),
                     xytext=(-r_apo / 1e6, 0),
                     arrowprops=dict(arrowstyle='->', color='C3', lw=2))
        ax1.text(-r_apo / 1e6 + 0.5, -arrow_scale * v_apo / v_peri - 1,
                 f'v_a={v_apo/1e3:.2f} km/s', fontsize=8, color='C3')
        ax1.set_aspect('equal')
        ax1.set_xlabel('x (Mm)')
        ax1.set_ylabel('y (Mm)')
        ax1.set_title(f'Elliptical orbit  e={e:.2f}  a={a/1e3:.0f} km\n'
                      f'T = {T/3600:.2f} h   Equal Δt = T/8 = {dt_wedge/60:.1f} min',
                      fontsize=10)
        ax1.legend(fontsize=8)
        ax2.plot(t_arr / 3600, np.rad2deg(nu_arr), 'C0', lw=2)
        ax2.axhline(180, ls=':', color='gray', alpha=0.5)
        ax2.set_xlabel('Time (hours)')
        ax2.set_ylabel('True anomaly ν (°)')
        ax2.set_title('Non-uniform angular motion\n'
                      '(faster near periapsis, slower near apoapsis)')
        ax2.set_xlim(0, T / 3600)
        plt.tight_layout()
        plt.show()

ecc_sl.observe(update_kepler, 'value')
a_sl.observe(update_kepler, 'value')
display(widgets.VBox([ecc_sl, a_sl]), out1)
update_kepler()

Output()

## 2 — Classical Orbital Elements

Six parameters uniquely define a Keplerian orbit:

| Element | Symbol | Description |
|:--|:--|:--|
| Semi-major axis | $a$ | Size of the orbit |
| Eccentricity | $e$ | Shape (0 = circle) |
| Inclination | $i$ | Tilt from equatorial plane |
| RAAN | $\Omega$ | Where the orbit crosses the equator northbound |
| Arg. of periapsis | $\omega$ | Angle from ascending node to periapsis |
| True anomaly | $\nu$ | Spacecraft position along the orbit |

The first two define the **shape**, the next three define the **orientation** in space, and the last locates the satellite.

In [3]:
out2 = widgets.Output()

sl_a2  = FloatSlider(value=20000, min=7000, max=42000, step=500, description='a (km)',
                     style={'description_width':'110px'}, layout=widgets.Layout(width='560px'),
                     continuous_update=False)
sl_e2  = FloatSlider(value=0.3, min=0.0, max=0.8, step=0.02, description='e',
                     style={'description_width':'110px'}, layout=widgets.Layout(width='560px'),
                     continuous_update=False)
sl_i2  = FloatSlider(value=30, min=0, max=180, step=2, description='i (°)',
                     style={'description_width':'110px'}, layout=widgets.Layout(width='560px'),
                     continuous_update=False)
sl_Om2 = FloatSlider(value=45, min=0, max=360, step=5, description='Ω (°)',
                     style={'description_width':'110px'}, layout=widgets.Layout(width='560px'),
                     continuous_update=False)
sl_w2  = FloatSlider(value=60, min=0, max=360, step=5, description='ω (°)',
                     style={'description_width':'110px'}, layout=widgets.Layout(width='560px'),
                     continuous_update=False)
sl_nu2 = FloatSlider(value=90, min=0, max=360, step=5, description='ν (°)',
                     style={'description_width':'110px'}, layout=widgets.Layout(width='560px'),
                     continuous_update=False)

def update_elements(change=None):
    a = sl_a2.value * 1e3
    e = sl_e2.value
    inc = np.deg2rad(sl_i2.value)
    Om = np.deg2rad(sl_Om2.value)
    w  = np.deg2rad(sl_w2.value)
    nu = np.deg2rad(sl_nu2.value)

    # rotation matrices
    def Rz(a):
        return np.array([[np.cos(a), -np.sin(a), 0],
                         [np.sin(a),  np.cos(a), 0],
                         [0, 0, 1]])
    def Rx(a):
        return np.array([[1, 0, 0],
                         [0, np.cos(a), -np.sin(a)],
                         [0, np.sin(a),  np.cos(a)]])

    Q = Rz(Om) @ Rx(inc) @ Rz(w)

    theta = np.linspace(0, 2 * np.pi, 500)
    r_orb = a * (1 - e**2) / (1 + e * np.cos(theta))
    x_pf = r_orb * np.cos(theta)
    y_pf = r_orb * np.sin(theta)
    z_pf = np.zeros_like(theta)
    orb_pf = np.vstack([x_pf, y_pf, z_pf])
    orb_eci = Q @ orb_pf

    # spacecraft position
    r_sc = a * (1 - e**2) / (1 + e * np.cos(nu))
    sc_pf = np.array([r_sc * np.cos(nu), r_sc * np.sin(nu), 0])
    sc_eci = Q @ sc_pf

    # periapsis direction
    peri_pf = np.array([a * (1 - e), 0, 0])
    peri_eci = Q @ peri_pf

    # ascending node direction
    an_dir = np.array([np.cos(Om), np.sin(Om), 0])

    with out2:
        clear_output(wait=True)
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')

        # equatorial plane
        eq_r = a * 1.3 / 1e6
        xx, yy = np.meshgrid(np.linspace(-eq_r, eq_r, 5),
                             np.linspace(-eq_r, eq_r, 5))
        ax.plot_surface(xx, yy, np.zeros_like(xx), alpha=0.08, color='gray')

        # Earth
        u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
        Re = R_E / 1e6
        ax.plot_surface(Re*np.cos(u)*np.sin(v), Re*np.sin(u)*np.sin(v),
                        Re*np.cos(v), color='#1a6fad', alpha=0.4)

        # orbit
        ax.plot(orb_eci[0]/1e6, orb_eci[1]/1e6, orb_eci[2]/1e6, 'C0', lw=2)

        # spacecraft
        ax.scatter(*sc_eci/1e6, color='C3', s=80, zorder=5, depthshade=False)

        # ascending node line
        an_len = a * 1.2 / 1e6
        ax.plot([0, an_dir[0]*an_len], [0, an_dir[1]*an_len], [0, 0],
               'C2', lw=2, ls='--', label=f'Ω = {sl_Om2.value:.0f}° (RAAN)')
        ax.scatter(*an_dir*an_len, color='C2', s=60, marker='^', depthshade=False)

        # periapsis line
        ax.plot([0, peri_eci[0]/1e6], [0, peri_eci[1]/1e6], [0, peri_eci[2]/1e6],
               'C1', lw=2, ls='--', label=f'ω = {sl_w2.value:.0f}° (arg peri)')

        # reference axes
        ax_len = eq_r * 0.7
        ax.quiver(0, 0, 0, ax_len, 0, 0, color='gray', alpha=0.5, arrow_length_ratio=0.08)
        ax.quiver(0, 0, 0, 0, ax_len, 0, color='gray', alpha=0.5, arrow_length_ratio=0.08)
        ax.quiver(0, 0, 0, 0, 0, ax_len, color='gray', alpha=0.5, arrow_length_ratio=0.08)
        ax.text(ax_len*1.1, 0, 0, 'X (♈)', fontsize=9, color='gray')
        ax.text(0, ax_len*1.1, 0, 'Y', fontsize=9, color='gray')
        ax.text(0, 0, ax_len*1.1, 'Z (pole)', fontsize=9, color='gray')

        ax.set_xlabel('X (Mm)')
        ax.set_ylabel('Y (Mm)')
        ax.set_zlabel('Z (Mm)')
        ax.set_title(f'Classical Orbital Elements\n'
                     f'a={sl_a2.value:.0f} km  e={e:.2f}  i={sl_i2.value:.0f}°  '
                     f'Ω={sl_Om2.value:.0f}°  ω={sl_w2.value:.0f}°  ν={sl_nu2.value:.0f}°',
                     fontsize=10)
        ax.legend(fontsize=8, loc='upper left')
        plt.tight_layout()
        plt.show()

for w in [sl_a2, sl_e2, sl_i2, sl_Om2, sl_w2, sl_nu2]:
    w.observe(update_elements, 'value')
display(widgets.VBox([sl_a2, sl_e2, sl_i2, sl_Om2, sl_w2, sl_nu2]), out2)
update_elements()

Output()

## 3 — Ground Tracks

A satellite's **ground track** is the trace of its sub-satellite point on Earth's surface.

Key relationships:
- The track is bounded in latitude by $\pm i$ (inclination)
- Earth rotates 360° in ~24 h → consecutive tracks shift **westward** by $\Delta\lambda = 360° \times T_{\text{orb}} / T_{\text{sid}}$
- **Repeat ground track** when $T_{\text{orb}} / T_{\text{sid}}$ is a rational number

| Orbit | Altitude | Inclination | Note |
|:--|:--|:--|:--|
| ISS | 408 km | 51.6° | Covers ±51.6° lat |
| Sun-sync | ~600 km | ~98° | Retrograde, J2-assisted |
| GEO | 35 786 km | 0° | Single point on equator |

In [4]:
out3 = widgets.Output()

alt_gt = FloatSlider(value=408, min=200, max=2000, step=10,
    description='Altitude (km)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
inc_gt = FloatSlider(value=51.6, min=0, max=110, step=0.5,
    description='Inclination (°)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
n_orbits = IntSlider(value=3, min=1, max=6, step=1,
    description='Orbits', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)

def update_ground(change=None):
    h = alt_gt.value * 1e3
    inc = np.deg2rad(inc_gt.value)
    r = R_E + h
    T_orb = 2 * np.pi * np.sqrt(r**3 / MU)
    n_orb = n_orbits.value
    T_sid = 86164.1  # sidereal day (s)
    dt = 10  # seconds
    t = np.arange(0, n_orb * T_orb, dt)
    n_mean = 2 * np.pi / T_orb
    # simple circular orbit: argument of latitude u = n*t
    u = n_mean * t
    # ECI position
    x_eci = r * (np.cos(u))
    y_eci = r * (np.cos(inc) * np.sin(u))
    z_eci = r * (np.sin(inc) * np.sin(u))
    # sub-satellite point
    lat = np.rad2deg(np.arcsin(np.sin(inc) * np.sin(u)))
    # longitude: account for Earth rotation
    lon = np.rad2deg(np.arctan2(np.cos(inc) * np.sin(u), np.cos(u))) - np.rad2deg(OMEGA_E * t)
    lon = ((lon + 180) % 360) - 180
    # detect wrapping
    dlon = np.diff(lon)
    wrap = np.where(np.abs(dlon) > 180)[0]
    shift_per_orbit = 360 * T_orb / T_sid

    with out3:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(13, 6))
        # grid lines for map
        for lat_line in range(-60, 90, 30):
            ax.axhline(lat_line, color='gray', lw=0.3, alpha=0.5)
        for lon_line in range(-150, 180, 30):
            ax.axvline(lon_line, color='gray', lw=0.3, alpha=0.5)
        ax.axhline(0, color='gray', lw=0.8, alpha=0.5)
        ax.axvline(0, color='gray', lw=0.8, alpha=0.5)
        # inclination bounds
        inc_deg = inc_gt.value
        ax.axhline(inc_deg, color='C3', ls='--', lw=1.2, alpha=0.6, label=f'±i = ±{inc_deg:.1f}°')
        ax.axhline(-inc_deg, color='C3', ls='--', lw=1.2, alpha=0.6)
        ax.fill_between([-180, 180], inc_deg, 90, alpha=0.05, color='C3')
        ax.fill_between([-180, 180], -90, -inc_deg, alpha=0.05, color='C3')
        # plot track segments
        segments = np.split(np.arange(len(lon)), wrap + 1)
        colors = plt.cm.viridis(np.linspace(0.2, 0.9, n_orb + 1))
        for seg in segments:
            if len(seg) < 2:
                continue
            orbit_num = int(t[seg[0]] / T_orb)
            ax.plot(lon[seg], lat[seg], color=colors[min(orbit_num, len(colors)-1)],
                    lw=1.5, alpha=0.8)
        # mark ascending nodes
        ascending = np.where((lat[:-1] < 0) & (lat[1:] >= 0))[0]
        for idx in ascending:
            ax.plot(lon[idx], lat[idx], '^', color='C2', ms=10, zorder=5)
        if len(ascending) > 0:
            ax.plot([], [], '^', color='C2', ms=8, label='Ascending node')
        # start point
        ax.plot(lon[0], lat[0], 'o', color='C0', ms=10, zorder=5, label='Start')
        ax.set_xlim(-180, 180)
        ax.set_ylim(-90, 90)
        ax.set_xlabel('Longitude (°)')
        ax.set_ylabel('Latitude (°)')
        ax.set_title(f'Ground Track — h = {alt_gt.value:.0f} km, i = {inc_deg:.1f}°, '
                     f'T = {T_orb/60:.1f} min\n'
                     f'Westward shift per orbit: {shift_per_orbit:.1f}°',
                     fontsize=10)
        ax.legend(fontsize=8, loc='lower left')
        plt.tight_layout()
        plt.show()

for w in [alt_gt, inc_gt, n_orbits]:
    w.observe(update_ground, 'value')
display(widgets.VBox([alt_gt, inc_gt, n_orbits]), out3)
update_ground()

Output()

## 4 — Orbital Perturbations: The J2 Effect

Earth is oblate — wider at the equator. The dominant perturbation is $J_2 = 1.0826 \times 10^{-3}$.

**RAAN precession** (regression of the ascending node):

$$\dot{\Omega} = -\frac{3}{2}\,n\,J_2\left(\frac{R_\oplus}{p}\right)^2 \cos i$$

**Argument of periapsis drift:**

$$\dot{\omega} = \frac{3}{2}\,n\,J_2\left(\frac{R_\oplus}{p}\right)^2 \left(2 - \frac{5}{2}\sin^2 i\right)$$

where $p = a(1-e^2)$ is the semi-latus rectum and $n = \sqrt{\mu/a^3}$.

**Sun-synchronous orbits** exploit this: by choosing $i \approx 97\text{–}99°$, $\dot{\Omega} \approx 0.9856°/\text{day}$, matching the Sun's apparent motion and keeping constant lighting conditions.

In [5]:
out4 = widgets.Output()

alt_j2 = FloatSlider(value=600, min=200, max=2000, step=10,
    description='Altitude (km)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
inc_j2 = FloatSlider(value=51.6, min=0, max=180, step=0.5,
    description='Inclination (°)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)

def j2_rates(h_m, inc_rad, e=0):
    a = R_E + h_m
    p = a * (1 - e**2)
    n = np.sqrt(MU / a**3)
    common = 1.5 * n * J2 * (R_E / p)**2
    Om_dot = -common * np.cos(inc_rad)           # rad/s
    w_dot  = common * (2 - 2.5 * np.sin(inc_rad)**2)  # rad/s
    return Om_dot, w_dot

def update_j2(change=None):
    h = alt_j2.value * 1e3
    inc = np.deg2rad(inc_j2.value)
    Om_dot, w_dot = j2_rates(h, inc)
    Om_dot_deg_day = np.rad2deg(Om_dot) * 86400
    w_dot_deg_day = np.rad2deg(w_dot) * 86400
    # sweep over inclination
    inc_arr = np.linspace(0, np.pi, 500)
    altitudes = [400e3, 600e3, 800e3, 1200e3]
    alt_labels = ['400 km', '600 km', '800 km', '1200 km']
    colors = ['C0', 'C1', 'C2', 'C3']

    with out4:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
        # RAAN drift
        for hi, lbl, c in zip(altitudes, alt_labels, colors):
            Od, _ = j2_rates(hi, inc_arr)
            ax1.plot(np.rad2deg(inc_arr), np.rad2deg(Od)*86400, color=c, lw=1.5, label=lbl)
        ax1.axhline(0.9856, ls=':', color='gray', lw=1, alpha=0.7)
        ax1.text(5, 1.2, 'Sun-sync = 0.9856°/day', fontsize=8, color='gray')
        ax1.axhline(-0.9856, ls=':', color='gray', lw=1, alpha=0.7)
        ax1.plot(inc_j2.value, Om_dot_deg_day, 'ko', ms=10, zorder=5)
        ax1.set_xlabel('Inclination (°)')
        ax1.set_ylabel('Ω̇ (°/day)')
        ax1.set_title('RAAN drift rate')
        ax1.legend(fontsize=8)
        ax1.set_xlim(0, 180)
        # ω drift
        for hi, lbl, c in zip(altitudes, alt_labels, colors):
            _, wd = j2_rates(hi, inc_arr)
            ax2.plot(np.rad2deg(inc_arr), np.rad2deg(wd)*86400, color=c, lw=1.5, label=lbl)
        ax2.axhline(0, ls='-', color='gray', lw=0.5)
        ax2.plot(inc_j2.value, w_dot_deg_day, 'ko', ms=10, zorder=5)
        # critical inclination
        i_crit = np.rad2deg(np.arcsin(np.sqrt(4/5)))  # ~63.4°
        ax2.axvline(i_crit, ls='--', color='C4', lw=1, alpha=0.7)
        ax2.text(i_crit + 1, ax2.get_ylim()[1]*0.8, f'i_crit = {i_crit:.1f}°\n(Molniya)',
                 fontsize=8, color='C4')
        ax2.set_xlabel('Inclination (°)')
        ax2.set_ylabel('ω̇ (°/day)')
        ax2.set_title('Arg. periapsis drift rate')
        ax2.legend(fontsize=8)
        ax2.set_xlim(0, 180)
        fig.suptitle(f'Selected: h = {alt_j2.value:.0f} km, i = {inc_j2.value:.1f}°  →  '
                     f'Ω̇ = {Om_dot_deg_day:.3f}°/day,  ω̇ = {w_dot_deg_day:.3f}°/day',
                     fontsize=10, y=1.02)
        plt.tight_layout()
        plt.show()

alt_j2.observe(update_j2, 'value')
inc_j2.observe(update_j2, 'value')
display(widgets.VBox([alt_j2, inc_j2]), out4)
update_j2()

Output()

## 5 — Bi-Elliptic Transfer

For large orbit ratio $r_2/r_1$, a **three-burn bi-elliptic** transfer can beat Hohmann.

The spacecraft first raises apoapsis to an intermediate radius $r_b > r_2$, circularises briefly, then lowers periapsis to $r_2$:

$$\Delta v_{\text{total}} = \Delta v_1 + \Delta v_2 + \Delta v_3$$

The **crossover** occurs at $r_2/r_1 \approx 11.94$ — above this ratio, bi-elliptic uses less $\Delta v$ than Hohmann, at the cost of much longer transfer time.

In [6]:
def hohmann_dv(r1, r2):
    a_t = (r1 + r2) / 2
    dv1 = np.sqrt(MU * (2/r1 - 1/a_t)) - np.sqrt(MU/r1)
    dv2 = np.sqrt(MU/r2) - np.sqrt(MU * (2/r2 - 1/a_t))
    tof = np.pi * np.sqrt(a_t**3 / MU)
    return abs(dv1) + abs(dv2), tof

def bielliptic_dv(r1, r2, rb):
    a1 = (r1 + rb) / 2
    a2 = (r2 + rb) / 2
    dv1 = np.sqrt(MU * (2/r1 - 1/a1)) - np.sqrt(MU/r1)
    dv2 = np.sqrt(MU * (2/rb - 1/a2)) - np.sqrt(MU * (2/rb - 1/a1))
    dv3 = np.sqrt(MU/r2) - np.sqrt(MU * (2/r2 - 1/a2))
    tof = np.pi * (np.sqrt(a1**3 / MU) + np.sqrt(a2**3 / MU))
    return abs(dv1) + abs(dv2) + abs(dv3), tof

out5 = widgets.Output()

r1_sl = FloatSlider(value=7000, min=6600, max=10000, step=100,
    description='r₁ (km)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
r2_sl = FloatSlider(value=105000, min=20000, max=200000, step=1000,
    description='r₂ (km)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
rb_sl = FloatSlider(value=300000, min=50000, max=500000, step=5000,
    description='r_b (km)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)

def update_bielliptic(change=None):
    r1 = r1_sl.value * 1e3
    r2 = r2_sl.value * 1e3
    rb = rb_sl.value * 1e3
    if rb < r2:
        rb = r2 * 1.1
    ratio = r2 / r1
    dv_h, tof_h = hohmann_dv(r1, r2)
    dv_b, tof_b = bielliptic_dv(r1, r2, rb)
    saving = (dv_h - dv_b) / dv_h * 100

    theta = np.linspace(0, 2 * np.pi, 400)
    scale = rb * 1.2

    with out5:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6),
                                        gridspec_kw={'width_ratios': [1.2, 0.8]})
        # orbits
        ax1.plot(r1*np.cos(theta)/scale, r1*np.sin(theta)/scale,
                 'C0', lw=1.5, ls='--', label=f'Orbit 1: r₁={r1/1e3:.0f} km')
        ax1.plot(r2*np.cos(theta)/scale, r2*np.sin(theta)/scale,
                 'C3', lw=1.5, ls='--', label=f'Orbit 2: r₂={r2/1e3:.0f} km')
        # Hohmann ellipse
        a_h = (r1 + r2) / 2
        b_h = np.sqrt(r1 * r2)
        cx_h = (r1 - r2) / 2
        t_arc = np.linspace(0, np.pi, 200)
        ax1.plot((cx_h + a_h*np.cos(t_arc))/scale, b_h*np.sin(t_arc)/scale,
                 'C2', lw=2, label=f'Hohmann Δv={dv_h/1e3:.3f} km/s')
        # Bi-elliptic: two half-ellipses
        a1_be = (r1 + rb) / 2
        b1_be = np.sqrt(r1 * rb)
        cx1 = (r1 - rb) / 2
        ax1.plot((cx1 + a1_be*np.cos(t_arc))/scale, b1_be*np.sin(t_arc)/scale,
                 'C1', lw=2, ls='-.', label=f'Bi-elliptic Δv={dv_b/1e3:.3f} km/s')
        a2_be = (r2 + rb) / 2
        b2_be = np.sqrt(r2 * rb)
        cx2 = (r2 - rb) / 2
        ax1.plot((cx2 + a2_be*np.cos(t_arc))/scale, -b2_be*np.sin(t_arc)/scale,
                 'C1', lw=2, ls='-.')
        # Earth
        ax1.add_patch(plt.Circle((0, 0), R_E/scale, color='#1a6fad', zorder=5))
        ax1.set_aspect('equal')
        ax1.set_xlim(-1.1, 1.1)
        ax1.set_ylim(-1.1, 1.1)
        ax1.axis('off')
        ax1.legend(fontsize=8, loc='lower right')
        ax1.set_title(f'Orbit ratio r₂/r₁ = {ratio:.1f}', fontsize=10)

        # comparison bars
        cats = ['Hohmann', 'Bi-elliptic']
        dvs = [dv_h/1e3, dv_b/1e3]
        clrs = ['C2', 'C1']
        bars = ax2.bar(cats, dvs, color=clrs, width=0.4, edgecolor='white', lw=1.5)
        for b, v in zip(bars, dvs):
            ax2.text(b.get_x()+b.get_width()/2, v + 0.02*max(dvs),
                     f'{v:.3f} km/s', ha='center', fontweight='bold', fontsize=10)
        winner = 'Bi-elliptic wins' if dv_b < dv_h else 'Hohmann wins'
        ax2.set_ylabel('Total Δv (km/s)')
        ax2.set_title(f'{winner}  ({saving:+.1f}% Δv)\n'
                      f'Hohmann: {tof_h/3600:.1f} h  |  Bi-elliptic: {tof_b/3600:.1f} h',
                      fontsize=9.5)
        ax2.set_ylim(0, max(dvs)*1.3)
        plt.tight_layout()
        plt.show()

for w in [r1_sl, r2_sl, rb_sl]:
    w.observe(update_bielliptic, 'value')
display(widgets.VBox([r1_sl, r2_sl, rb_sl]), out5)
update_bielliptic()

Output()

## 6 — Gravity Assists (Planetary Flyby)

A spacecraft approaching a planet at hyperbolic excess velocity $v_\infty$ follows a **hyperbolic trajectory** around the planet. The trajectory bends the velocity vector by the **turn angle**:

$$\delta = 2\arcsin\!\left(\frac{1}{1 + r_p\,v_\infty^2/\mu_p}\right)$$

In the planet's frame, the speed is unchanged — only the direction changes. But in the Sun's frame, the planet's orbital velocity adds a net $\Delta v$ for free:

$$\Delta v_{\text{free}} = 2\,v_\infty\,\sin\!\left(\frac{\delta}{2}\right)$$

This is how Voyager 1 & 2 reached Jupiter and Saturn with far less fuel than a direct transfer.

In [ ]:
out6 = widgets.Output()

vinf_sl = FloatSlider(value=10, min=1, max=30, step=0.5,
    description='v∞ (km/s)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
rp_sl = FloatSlider(value=2.0, min=1.05, max=10, step=0.05,
    description='rₚ (planet radii)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
planet_dd = widgets.Dropdown(
    options=[('Earth', (R_E, MU, 29.78)),
             ('Venus', (6.052e6, 3.249e14, 35.02)),
             ('Mars',  (3.390e6, 4.283e13, 24.07)),
             ('Jupiter', (6.991e7, 1.267e17, 13.07)),
             ('Saturn', (5.823e7, 3.793e16, 9.68))],
    value=(6.991e7, 1.267e17, 13.07),
    description='Planet:', style={'description_width':'110px'},
    layout=widgets.Layout(width='360px'))

def update_flyby(change=None):
    v_inf = vinf_sl.value * 1e3
    R_planet, mu_planet, v_planet_kms = planet_dd.value
    v_planet = v_planet_kms * 1e3
    r_p = rp_sl.value * R_planet
    # hyperbolic orbit params
    a_hyp = mu_planet / v_inf**2
    e_hyp = 1 + r_p * v_inf**2 / mu_planet
    delta = 2 * np.arcsin(1 / e_hyp)  # turn angle
    dv_free = 2 * v_inf * np.sin(delta / 2)
    # hyperbolic trajectory
    nu_max = np.arccos(-1 / e_hyp) - 0.05
    nu = np.linspace(-nu_max, nu_max, 600)
    r_hyp = a_hyp * (e_hyp**2 - 1) / (1 + e_hyp * np.cos(nu))
    x_hyp = r_hyp * np.cos(nu)
    y_hyp = r_hyp * np.sin(nu)
    # velocity vectors in Sun frame
    # incoming: planet vel + v_inf (from left)
    # outgoing: planet vel + v_inf (rotated by delta)
    v_in_sun = np.array([v_planet - v_inf * np.cos(delta/2), -v_inf * np.sin(delta/2)])
    v_out_sun = np.array([v_planet + v_inf * np.cos(delta/2) * np.cos(delta) - v_inf * np.sin(delta/2) * np.sin(delta),
                          v_inf * np.sin(delta/2)])
    # simpler: approach from right, leave rotated
    v_in_planet = np.array([-v_inf, 0])  # incoming in planet frame
    # after flyby, rotated by delta
    v_out_planet = v_inf * np.array([-np.cos(delta), np.sin(delta)])
    # Sun frame
    v_p = np.array([0, v_planet])  # planet moves in +y in sun frame
    v_in_sun = v_p + v_in_planet
    v_out_sun = v_p + v_out_planet

    with out6:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
        # planet frame trajectory
        ax1.plot(x_hyp / R_planet, y_hyp / R_planet, 'C0', lw=2)
        circle = plt.Circle((0, 0), 1.0, color='#e67e22', alpha=0.5, zorder=5)
        ax1.add_patch(circle)
        ax1.plot(0, 0, 'o', color='#e67e22', ms=3, zorder=6)
        # periapsis
        ax1.plot(r_p / R_planet, 0, 'o', color='C3', ms=8, zorder=5)
        ax1.annotate(f'Periapsis\n{r_p/R_planet:.1f} Rp = {r_p/1e3:.0f} km',
                     (r_p / R_planet, 0), xytext=(r_p/R_planet + 2, 2),
                     fontsize=8, color='C3',
                     arrowprops=dict(arrowstyle='->', color='C3'))
        # asymptotes
        x_far = x_hyp[0] / R_planet
        y_far = y_hyp[0] / R_planet
        ax1.annotate('', xy=(x_far, y_far - 2),
                     xytext=(x_far, y_far),
                     arrowprops=dict(arrowstyle='->', color='C0', lw=2))
        ax1.annotate('', xy=(x_hyp[-1]/R_planet, y_hyp[-1]/R_planet + 2),
                     xytext=(x_hyp[-1]/R_planet, y_hyp[-1]/R_planet),
                     arrowprops=dict(arrowstyle='->', color='C2', lw=2))
        ax1.set_aspect('equal')
        lim = max(abs(x_hyp).max(), abs(y_hyp).max()) / R_planet * 1.1
        ax1.set_xlim(-lim*0.3, lim)
        ax1.set_ylim(-lim, lim)
        ax1.set_xlabel('x (planet radii)')
        ax1.set_ylabel('y (planet radii)')
        ax1.set_title(f'Hyperbolic flyby — Planet frame\n'
                      f'e = {e_hyp:.2f}  δ = {np.rad2deg(delta):.1f}°',
                      fontsize=10)

        # Sun frame velocity vectors
        v_scale = 1e3
        ax2.arrow(0, 0, v_in_sun[0]/v_scale, v_in_sun[1]/v_scale,
                  head_width=0.3, head_length=0.2, fc='C0', ec='C0', lw=2,
                  label=f'v_in = {np.linalg.norm(v_in_sun)/1e3:.1f} km/s')
        ax2.arrow(0, 0, v_out_sun[0]/v_scale, v_out_sun[1]/v_scale,
                  head_width=0.3, head_length=0.2, fc='C2', ec='C2', lw=2,
                  label=f'v_out = {np.linalg.norm(v_out_sun)/1e3:.1f} km/s')
        ax2.arrow(0, 0, v_p[0]/v_scale, v_p[1]/v_scale,
                  head_width=0.3, head_length=0.2, fc='#e67e22', ec='#e67e22', lw=2,
                  label=f'v_planet = {v_planet/1e3:.1f} km/s')
        all_v = np.array([v_in_sun, v_out_sun, v_p]) / v_scale
        vmax = np.abs(all_v).max() * 1.4
        ax2.set_xlim(-vmax, vmax)
        ax2.set_ylim(-vmax*0.5, vmax*1.2)
        ax2.set_aspect('equal')
        ax2.set_xlabel('vx (km/s)')
        ax2.set_ylabel('vy (km/s)')
        ax2.set_title(f'Velocity vectors — Sun frame\n'
                      f'Free Δv = {dv_free/1e3:.2f} km/s',
                      fontsize=10)
        ax2.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

for w in [vinf_sl, rp_sl, planet_dd]:
    w.observe(update_flyby, 'value')
display(widgets.VBox([planet_dd, vinf_sl, rp_sl]), out6)
update_flyby()

Output()

## 7 — Interplanetary Transfers: Earth–Mars Geometry

A Hohmann-like transfer from Earth to Mars requires launching when the relative geometry is right — the **launch window**.

Earth orbit: $a_E = 1\,\text{AU}$, $T_E = 365.25\,\text{d}$  
Mars orbit: $a_M = 1.524\,\text{AU}$, $T_M = 687\,\text{d}$

**Synodic period** (time between consecutive launch windows):

$$T_{\text{syn}} = \frac{1}{|1/T_E - 1/T_M|} \approx 780\,\text{days} \approx 2.14\,\text{years}$$

The transfer ellipse has $a_t = (a_E + a_M)/2$. Mars must be ~44° ahead of Earth at departure so it arrives at apoapsis when the spacecraft does.

In [8]:
out7 = widgets.Output()

dep_sl = FloatSlider(value=0, min=0, max=360, step=2,
    description='Earth pos (°)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
mars_lead = FloatSlider(value=44, min=0, max=180, step=1,
    description='Mars lead (°)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)

R_EARTH_ORB = 1.0  # AU
R_MARS_ORB = 1.524  # AU
T_EARTH = 365.25  # days
T_MARS = 687.0  # days

def update_transfer(change=None):
    theta_E = np.deg2rad(dep_sl.value)
    theta_M_dep = theta_E + np.deg2rad(mars_lead.value)
    # transfer ellipse
    a_t = (R_EARTH_ORB + R_MARS_ORB) / 2  # AU
    tof = np.pi * np.sqrt((a_t * AU)**3 / MU_S) / 86400  # days
    # Mars position at arrival
    omega_M = 2 * np.pi / T_MARS  # rad/day
    theta_M_arr = theta_M_dep + omega_M * tof
    # delta-v
    v_E = np.sqrt(MU_S / (R_EARTH_ORB * AU))
    v_dep = np.sqrt(MU_S * (2 / (R_EARTH_ORB * AU) - 1 / (a_t * AU)))
    v_M = np.sqrt(MU_S / (R_MARS_ORB * AU))
    v_arr = np.sqrt(MU_S * (2 / (R_MARS_ORB * AU) - 1 / (a_t * AU)))
    dv1 = abs(v_dep - v_E)
    dv2 = abs(v_M - v_arr)
    # check if Mars is at right position at arrival
    # ideal: theta_M_arr should be at theta_E + 180° (apoapsis of transfer)
    arr_angle = (theta_E + np.pi) % (2 * np.pi)
    mars_arr_angle = theta_M_arr % (2 * np.pi)
    miss_angle = np.rad2deg(mars_arr_angle - arr_angle)
    miss_angle = ((miss_angle + 180) % 360) - 180

    theta_full = np.linspace(0, 2 * np.pi, 400)

    with out7:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6),
                                        gridspec_kw={'width_ratios': [1.2, 0.8]})
        # Sun
        ax1.plot(0, 0, 'o', color='#f1c40f', ms=20, zorder=5)
        # orbits
        ax1.plot(R_EARTH_ORB*np.cos(theta_full), R_EARTH_ORB*np.sin(theta_full),
                 'C0', lw=1, ls='--', alpha=0.5)
        ax1.plot(R_MARS_ORB*np.cos(theta_full), R_MARS_ORB*np.sin(theta_full),
                 'C3', lw=1, ls='--', alpha=0.5)
        # Earth at departure
        ax1.plot(R_EARTH_ORB*np.cos(theta_E), R_EARTH_ORB*np.sin(theta_E),
                 'o', color='C0', ms=14, zorder=5, label='Earth (departure)')
        # Mars at departure
        ax1.plot(R_MARS_ORB*np.cos(theta_M_dep), R_MARS_ORB*np.sin(theta_M_dep),
                 's', color='C3', ms=10, zorder=5, alpha=0.4, label='Mars (departure)')
        # Mars at arrival
        ax1.plot(R_MARS_ORB*np.cos(theta_M_arr), R_MARS_ORB*np.sin(theta_M_arr),
                 's', color='C3', ms=14, zorder=5, label='Mars (arrival)')
        # transfer ellipse (half)
        b_t = np.sqrt(R_EARTH_ORB * R_MARS_ORB)
        cx_t = (R_EARTH_ORB - R_MARS_ORB) / 2
        t_arc = np.linspace(0, np.pi, 200)
        # rotate transfer to align with Earth departure
        xe = cx_t + a_t * np.cos(t_arc)
        ye = b_t * np.sin(t_arc)
        # rotate by theta_E
        xr = xe * np.cos(theta_E) - ye * np.sin(theta_E)
        yr = xe * np.sin(theta_E) + ye * np.cos(theta_E)
        ax1.plot(xr, yr, 'C1', lw=2.5, label='Transfer orbit')
        ax1.set_aspect('equal')
        ax1.set_xlim(-2, 2)
        ax1.set_ylim(-2, 2)
        ax1.legend(fontsize=8, loc='lower left')
        ax1.set_title('Earth–Mars transfer geometry', fontsize=10)
        ax1.set_xlabel('x (AU)')
        ax1.set_ylabel('y (AU)')

        # info panel
        info = [
            f'Transfer semi-major axis: {a_t:.3f} AU',
            f'Time of flight: {tof:.0f} days ({tof/30.44:.1f} months)',
            f'Departure Δv: {dv1/1e3:.2f} km/s',
            f'Arrival Δv: {dv2/1e3:.2f} km/s',
            f'Total Δv: {(dv1+dv2)/1e3:.2f} km/s',
            f'',
            f'Mars lead angle: {mars_lead.value:.0f}°',
            f'Miss angle at arrival: {miss_angle:+.1f}°',
            f'(0° = perfect encounter)',
            f'',
            f'Synodic period: {1/abs(1/T_EARTH - 1/T_MARS):.0f} days',
            f'Optimal lead angle: ~44°',
        ]
        ax2.axis('off')
        for i, line in enumerate(info):
            color = 'C2' if 'perfect' in line.lower() else 'k'
            weight = 'bold' if 'Total' in line or 'Miss' in line else 'normal'
            ax2.text(0.05, 0.92 - i*0.075, line, fontsize=11, transform=ax2.transAxes,
                     fontweight=weight, color=color if 'Miss' in line else 'k',
                     fontfamily='monospace')
        quality = abs(miss_angle)
        if quality < 10:
            grade_txt, grade_clr = 'GOOD WINDOW', 'C2'
        elif quality < 30:
            grade_txt, grade_clr = 'MARGINAL', 'C1'
        else:
            grade_txt, grade_clr = 'BAD WINDOW', 'C3'
        ax2.text(0.5, 0.02, grade_txt, fontsize=16, ha='center',
                 transform=ax2.transAxes, color=grade_clr, fontweight='bold',
                 bbox=dict(fc='white', ec=grade_clr, lw=2, pad=6))
        plt.tight_layout()
        plt.show()

dep_sl.observe(update_transfer, 'value')
mars_lead.observe(update_transfer, 'value')
display(widgets.VBox([dep_sl, mars_lead]), out7)
update_transfer()

Output()

## 8 — Lagrange Points

In the **circular restricted three-body problem** (CR3BP), five equilibrium points exist where the gravitational and centrifugal forces balance.

The mass ratio $\mu = m_2 / (m_1 + m_2)$ determines the geometry.

| Point | Location | Stability | Missions |
|:--|:--|:--|:--|
| L1 | Between bodies | Unstable | SOHO, DSCOVR |
| L2 | Beyond smaller body | Unstable | JWST, Gaia |
| L3 | Opposite smaller body | Unstable | — |
| L4, L5 | 60° ahead/behind | Stable (if $\mu < 0.0385$) | Trojan asteroids |

The **effective potential** in the rotating frame:

$$U_{\text{eff}} = -\frac{1-\mu}{r_1} - \frac{\mu}{r_2} - \frac{1}{2}(x^2 + y^2)$$

In [ ]:
out8 = widgets.Output()

mu_sl = widgets.FloatLogSlider(value=3.003e-6, base=10, min=-6, max=-1, step=0.1,
    description='μ = m₂/(m₁+m₂)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
system_dd = widgets.Dropdown(
    options=[('Sun–Earth', 3.003e-6), ('Earth–Moon', 0.01215), ('Sun–Jupiter', 9.537e-4)],
    value=3.003e-6,
    description='System:', style={'description_width':'110px'},
    layout=widgets.Layout(width='360px'))

def find_L1(mu):
    def eq(x): return x - (1-mu)/(x+mu)**2 + mu/(x-1+mu)**2
    return brentq(eq, -mu+0.01, 1-mu-0.01)

def find_L2(mu):
    def eq(x): return x - (1-mu)/(x+mu)**2 - mu/(x-1+mu)**2
    return brentq(eq, 1-mu+0.01, 2.0)

def find_L3(mu):
    def eq(x): return x + (1-mu)/(x+mu)**2 + mu/(x-1+mu)**2
    return brentq(eq, -2.0, -mu-0.01)

def sync_mu(change):
    mu_sl.value = system_dd.value

def update_lagrange(change=None):
    mu = mu_sl.value
    # positions: m1 at (-mu, 0), m2 at (1-mu, 0)
    x1, y1 = -mu, 0
    x2, y2 = 1 - mu, 0
    try:
        xL1 = find_L1(mu)
        xL2 = find_L2(mu)
        xL3 = find_L3(mu)
    except:
        xL1, xL2, xL3 = 0.5, 1.5, -1.0
    xL4, yL4 = 0.5 - mu, np.sqrt(3)/2
    xL5, yL5 = 0.5 - mu, -np.sqrt(3)/2

    # effective potential
    N = 400
    xg = np.linspace(-1.8, 1.8, N)
    yg = np.linspace(-1.8, 1.8, N)
    X, Y = np.meshgrid(xg, yg)
    r1 = np.sqrt((X - x1)**2 + Y**2)
    r2 = np.sqrt((X - x2)**2 + Y**2)
    r1 = np.maximum(r1, 1e-4)
    r2 = np.maximum(r2, 1e-4)
    U = -(1 - mu) / r1 - mu / r2 - 0.5 * (X**2 + Y**2)

    with out8:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 8))
        # effective potential contours
        levels = np.linspace(U.min(), U.max()*0.5, 40)
        ax.contourf(X, Y, U, levels=levels, cmap='RdYlBu_r', alpha=0.6)
        ax.contour(X, Y, U, levels=levels, colors='gray', linewidths=0.3, alpha=0.5)
        # zero-velocity curves at Lagrange point energies
        for lx, ly, lbl in [(xL1, 0, 'L1'), (xL2, 0, 'L2')]:
            r1_L = np.sqrt((lx - x1)**2 + ly**2)
            r2_L = np.sqrt((lx - x2)**2 + ly**2)
            U_L = -(1-mu)/r1_L - mu/r2_L - 0.5*(lx**2 + ly**2)
            ax.contour(X, Y, U, levels=[U_L], colors='k', linewidths=1, alpha=0.4)
        # bodies
        ax.plot(x1, y1, 'o', color='#f1c40f', ms=18, zorder=5, label='m₁ (primary)')
        ax.plot(x2, y2, 'o', color='#3498db', ms=10, zorder=5, label='m₂ (secondary)')
        # Lagrange points
        lpts = [(xL1, 0, 'L1'), (xL2, 0, 'L2'), (xL3, 0, 'L3'),
                (xL4, yL4, 'L4'), (xL5, yL5, 'L5')]
        for lx, ly, lbl in lpts:
            color = 'C3' if lbl in ['L1','L2','L3'] else 'C2'
            ax.plot(lx, ly, 'x', color=color, ms=12, mew=2.5, zorder=6)
            ax.annotate(lbl, (lx, ly), xytext=(8, 8), textcoords='offset points',
                        fontsize=11, fontweight='bold', color=color)
        # annotations for real missions
        if abs(mu - 3.003e-6) < 1e-4:
            ax.annotate('JWST, Gaia', (xL2, 0), xytext=(15, -20),
                        textcoords='offset points', fontsize=8, color='gray',
                        arrowprops=dict(arrowstyle='->', color='gray'))
            ax.annotate('SOHO, DSCOVR', (xL1, 0), xytext=(-60, 15),
                        textcoords='offset points', fontsize=8, color='gray',
                        arrowprops=dict(arrowstyle='->', color='gray'))
        ax.set_aspect('equal')
        ax.set_xlabel('x (rotating frame)')
        ax.set_ylabel('y (rotating frame)')
        ax.set_title(f'Lagrange Points  —  μ = {mu:.2e}\n'
                     f'L1 = ({xL1:.4f}, 0)  L2 = ({xL2:.4f}, 0)  '
                     f'L4 = ({xL4:.4f}, {yL4:.4f})', fontsize=10)
        ax.legend(fontsize=8, loc='upper right')
        plt.tight_layout()
        plt.show()

system_dd.observe(sync_mu, 'value')
mu_sl.observe(update_lagrange, 'value')
display(widgets.VBox([system_dd, mu_sl]), out8)
update_lagrange()

Output()

## 9 — Atmospheric Re-Entry

A spacecraft returning from orbit must dissipate enormous kinetic energy as heat.

The key parameters are:
- **Entry velocity** $V_e$: LEO ~7.8 km/s, lunar return ~11 km/s
- **Entry angle** $\gamma_e$: shallow → skip off atmosphere; steep → extreme g-loads
- **Ballistic coefficient** $\beta = m/(C_D A)$: low $\beta$ (large, light) → decelerates high; high $\beta$ → punches deeper

The exponential atmosphere model: $\rho(h) = \rho_0\,e^{-h/H}$ with scale height $H \approx 7.5$ km.

**Peak deceleration:**

$$a_{\max} = \frac{V_e^2 \sin\gamma_e}{2eH}$$

**Stagnation-point heat flux** (Chapman's approximation):

$$\dot{q} \propto \sqrt{\frac{\rho}{R_n}}\,V^3$$

In [12]:
out9 = widgets.Output()

ve_sl = FloatSlider(value=7.8, min=5, max=13, step=0.1,
    description='V_entry (km/s)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
gamma_sl = FloatSlider(value=-5, min=-15, max=-0.5, step=0.5,
    description='γ_entry (°)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
beta_sl = FloatSlider(value=200, min=20, max=1000, step=10,
    description='β (kg/m²)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)

def update_reentry(change=None):
    Ve = ve_sl.value * 1e3
    gamma_e = np.deg2rad(gamma_sl.value)
    beta = beta_sl.value
    H_scale = 7500  # m
    rho0 = 1.225  # kg/m^3
    R_nose = 1.0  # m (nose radius for heating)
    h_entry = 120e3  # m
    # integrate simple ballistic entry equations
    # state: [V, gamma, h, s]
    def reentry_ode(t, y):
        V, gam, h, s = y
        if h < 0 or V < 10:
            return [0, 0, 0, 0]
        rho = rho0 * np.exp(-h / H_scale)
        D = 0.5 * rho * V**2 / beta  # deceleration = D/m = 0.5*rho*V^2*Cd*A/m
        g = G0 * (R_E / (R_E + h))**2
        dVdt = -D - g * np.sin(gam)
        dgamdt = (V / (R_E + h) - g / V) * np.cos(gam)
        dhdt = V * np.sin(gam)
        dsdt = V * np.cos(gam) * R_E / (R_E + h)
        return [dVdt, dgamdt, dhdt, dsdt]
    def hit_ground(t, y): return y[2]  # h = 0
    hit_ground.terminal = True
    hit_ground.direction = -1
    y0 = [Ve, gamma_e, h_entry, 0]
    sol = solve_ivp(reentry_ode, [0, 3000], y0, max_step=0.5,
                    events=hit_ground, dense_output=True)
    t = sol.t
    V = sol.y[0]
    gam = sol.y[1]
    h = sol.y[2]
    s = sol.y[3]
    rho_traj = rho0 * np.exp(-h / H_scale)
    decel = 0.5 * rho_traj * V**2 / beta / G0  # in g's
    q_dot = 1.83e-4 * np.sqrt(rho_traj / R_nose) * (V)**3  # W/m^2 (Chapman approx)

    with out9:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        # trajectory
        ax = axes[0]
        ax.plot(s / 1e3, h / 1e3, 'C0', lw=2)
        ax.plot(s[0] / 1e3, h[0] / 1e3, 'o', color='C0', ms=8, label='Entry')
        ax.set_xlabel('Downrange (km)')
        ax.set_ylabel('Altitude (km)')
        ax.set_title('Re-entry trajectory')
        ax.set_ylim(0, 130)
        ax.legend(fontsize=8)
        # g-load
        ax = axes[1]
        ax.plot(decel, h / 1e3, 'C3', lw=2)
        max_g = decel.max()
        max_g_h = h[np.argmax(decel)] / 1e3
        ax.plot(max_g, max_g_h, 'o', color='C3', ms=8)
        ax.annotate(f'{max_g:.1f}g at {max_g_h:.0f} km', (max_g, max_g_h),
                    xytext=(max_g*1.1, max_g_h+5), fontsize=9, color='C3',
                    arrowprops=dict(arrowstyle='->', color='C3'))
        ax.axvline(10, ls=':', color='gray', alpha=0.5)
        ax.text(10.5, 100, 'Human limit ~10g', fontsize=8, color='gray')
        ax.set_xlabel('Deceleration (g)')
        ax.set_ylabel('Altitude (km)')
        ax.set_title('G-load profile')
        ax.set_ylim(0, 130)
        # heating
        ax = axes[2]
        ax.plot(q_dot / 1e6, h / 1e3, 'C1', lw=2)
        max_q = q_dot.max()
        max_q_h = h[np.argmax(q_dot)] / 1e3
        ax.plot(max_q / 1e6, max_q_h, 'o', color='C1', ms=8)
        ax.annotate(f'{max_q/1e6:.1f} MW/m² at {max_q_h:.0f} km',
                    (max_q / 1e6, max_q_h),
                    xytext=(max_q/1e6*0.7, max_q_h+8), fontsize=9, color='C1',
                    arrowprops=dict(arrowstyle='->', color='C1'))
        ax.set_xlabel('Heat flux (MW/m²)')
        ax.set_ylabel('Altitude (km)')
        ax.set_title('Stagnation heating')
        ax.set_ylim(0, 130)
        fig.suptitle(f'Ballistic Re-Entry — V_e={Ve/1e3:.1f} km/s  '
                     f'γ={gamma_sl.value:.1f}°  β={beta:.0f} kg/m²\n'
                     f'Peak: {max_g:.1f}g at {max_g_h:.0f} km  |  '
                     f'{max_q/1e6:.1f} MW/m² at {max_q_h:.0f} km  |  '
                     f'Range: {s[-1]/1e3:.0f} km',
                     fontsize=10, y=1.04)
        plt.tight_layout()
        plt.show()

for w in [ve_sl, gamma_sl, beta_sl]:
    w.observe(update_reentry, 'value')
display(widgets.VBox([ve_sl, gamma_sl, beta_sl]), out9)
update_reentry()

Output()